# One-run runtime check
Run each of the six active embedding methods **once** on the full CMiO, North Batten and synthetic datasets, with seed 42 and 10 neighbours.

Times are wall-clock seconds for the embedding call only. Loading, data generation, scaling, clustering and evaluation are excluded. There is no warm-up or repeated-seed loop: the first method using a compiled function may include compilation time. SpaSNE includes the adapter and WSL startup on Windows. These are single-run timings, not averaged benchmarks.

Run all cells from top to bottom in a fresh kernel using the project environment. SpaSNE needs its existing compiled executable and WSL on Windows.

## 1. Setup
Use one active Numba thread, as in the results notebooks. Find the project folder from the current notebook location.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

# Set the pool size before imports, or keep it if Numba is already loaded.
if "numba" not in sys.modules:
    os.environ["NUMBA_NUM_THREADS"] = "1"

import numba
import numpy as np
import pandas as pd
import pacmap
import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

numba.set_num_threads(1)

ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [folder, folder / "sc-pacmap-reproducibility"]:
        if (candidate / "code" / "sc_pacmap" / "sc_pacmap.py").is_file():
            ROOT = candidate
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project folder.")

sys.path.insert(0, str(ROOT / "code" / "sc_pacmap"))
sys.path.insert(0, str(ROOT / "code" / "spasne"))
from sc_pacmap import PaCMAPSettings, SCPaCMAPConfig, run_sc_pacmap
from spasne_adapter import python_spasne

SEED = 42
N_NEIGHBORS = 10

## 2. Runtime checker
Standardize inputs before starting the timer. Each method gets the same rows and the settings used in the results notebooks. Coordinate baselines use XYZ for real data and XY for synthetic data. SC-PaCMAP and SpaSNE receive features and raw coordinates separately.

The checker prints each completed time and returns a small table. A failed method is recorded with its error so the remaining methods can still run. Numerical thread pools are limited to one during fitting.

In [ ]:
def check_runtime(dataset_name, data, feature_columns, coordinate_columns):
    features = StandardScaler().fit_transform(data[feature_columns]).astype(np.float32)
    coordinates = data[coordinate_columns].to_numpy(dtype=float)
    scaled_coordinates = StandardScaler().fit_transform(coordinates)
    combined = np.column_stack([features, scaled_coordinates])
    localmap_input = np.ascontiguousarray(combined, dtype=np.float64)

    pacmap_options = dict(
        n_components=2, n_neighbors=N_NEIGHBORS, MN_ratio=0.5, FP_ratio=2.0,
        distance="euclidean", lr=1.0, num_iters=(100, 100, 250),
        apply_pca=True, random_state=SEED, save_tree=False, knn_backend="faiss",
    )
    sc_settings = PaCMAPSettings(
        n_components=2, n_neighbors=N_NEIGHBORS, mn_ratio=0.5, fp_ratio=2.0,
        learning_rate=1.0, num_iters=(100, 100, 250), random_state=SEED,
        knn_backend="faiss", verbose=False,
    )
    coordinate_label = " + " + "".join(coordinate_columns).upper()
    if coordinate_columns == ["x_m", "y_m"]:
        coordinate_label = " + XY"
    methods = ["UMAP", "PaCMAP", "LocalMAP", "PaCMAP feature-only", "SC-PaCMAP", "SpaSNE"]
    rows = []

    with threadpool_limits(limits=1):
        for method in methods:
            name = method + coordinate_label if method in ["UMAP", "PaCMAP", "LocalMAP"] else method
            print(f"{dataset_name}: starting {name}", flush=True)
            seconds = np.nan
            error = ""
            try:
                start = perf_counter()
                if method == "UMAP":
                    embedding = umap.UMAP(
                        n_neighbors=N_NEIGHBORS, n_components=2, metric="euclidean",
                        learning_rate=1.0, random_state=SEED, n_jobs=1,
                    ).fit_transform(combined)
                elif method == "PaCMAP":
                    embedding = pacmap.PaCMAP(**pacmap_options).fit_transform(combined, init="pca", save_pairs=False)
                elif method == "LocalMAP":
                    embedding = pacmap.LocalMAP(**pacmap_options, low_dist_thres=10).fit_transform(localmap_input, init="pca", save_pairs=False)
                elif method == "PaCMAP feature-only":
                    embedding = pacmap.PaCMAP(**pacmap_options).fit_transform(features, init="pca", save_pairs=False)
                elif method == "SC-PaCMAP":
                    result, diagnostics = run_sc_pacmap(features, coordinates, sc_settings, config=SCPaCMAPConfig())
                    embedding = result.embedding
                elif method == "SpaSNE":
                    embedding = python_spasne(
                        features, coordinates, random_state=SEED, alpha=10.0, beta=5.0,
                        exaggeration=4, perplexity=50, theta=0.5, max_iter=1000,
                        use_pca=True, spasne_dir=ROOT / "code" / "spasne" / "spasne_original",
                    )
                seconds = perf_counter() - start
                if embedding.shape != (len(data), 2) or not np.isfinite(embedding).all():
                    raise ValueError("The embedding must have two finite columns.")
                print(f"  {seconds:.2f} seconds", flush=True)
            except Exception as exc:
                seconds = np.nan
                error = str(exc)
                print(f"  Failed: {error}", flush=True)
            rows.append(dict(dataset=dataset_name, method=name, samples=len(data),
                             features=len(feature_columns), seed=SEED, seconds=seconds, error=error))
    return pd.DataFrame(rows)

## 3. CMiO
Load the prepared data and use the same 35 geochemical features as the CMiO results notebook.

In [ ]:
cmio = pd.read_csv(ROOT / "data" / "processed" / "cmio_sc_pacmap_ready.csv")
cmio_features = [
    "Y_PPM", "V_PPM", "LA_PPM", "U_PPM", "SR_PPM", "SN_PPM", "RB_PPM",
    "TH_PPM", "CS_PPM", "CE_PPM", "NB_PPM", "GA_PPM", "W_PPM", "HF_PPM",
    "TA_PPM", "ZR_PPM", "CR_PPM", "PR_PPM", "ND_PPM", "ER_PPM", "EU_PPM",
    "SM_PPM", "YB_PPM", "TB_PPM", "HO_PPM", "GD_PPM", "DY_PPM", "LU_PPM",
    "BA_PPM", "SIO2_WT_PERCENT", "TIO2_WT_PERCENT", "NA2O_WT_PERCENT",
    "K2O_WT_PERCENT", "AL2O3_WT_PERCENT", "TM_PPM",
]
cmio_times = check_runtime("CMiO", cmio, cmio_features, ["x", "y", "z"])
display(cmio_times.round({"seconds": 2}))

## 4. North Batten
Use the prepared data, all PPM/PCT features and raw XYZ coordinates.

In [ ]:
north_batten = pd.read_csv(ROOT / "data" / "processed" / "north_batten_sc_pacmap_ready.csv")
north_features = [column for column in north_batten.columns if column.endswith(("_PPM", "_PCT"))]
north_times = check_runtime("North Batten", north_batten, north_features, ["x", "y", "z"])
display(north_times.round({"seconds": 2}))

## 5. Synthetic data
Recreate the existing seed-42 synthetic benchmark: a 48 by 48 grid, 16 patches, four repeated facies and 50 features (10 spatial, 20 confounders, 20 noise). Generation happens outside the timer.

In [ ]:
rng = np.random.default_rng(SEED)
grid_x, grid_y = np.meshgrid(np.arange(48), np.arange(48))
x_index = grid_x.ravel()
y_index = grid_y.ravel()
sample_count = len(x_index)
indices = np.column_stack([x_index, y_index]).astype(float)
coordinates = 25.0 * (indices + rng.normal(0.0, 0.04, size=indices.shape))
facies = (x_index // 12 + y_index // 12) % 4
prototypes = np.array([[-1, -1, -1], [-1, 1, 1], [1, -1, 1], [1, 1, -1]], dtype=float)

spatial_columns = []
for index, sigma in enumerate(np.linspace(0.15, 0.95, 10)):
    signal = prototypes[facies, index % 3]
    spatial_columns.append(signal + sigma * rng.normal(size=sample_count))
spatial = np.column_stack(spatial_columns)

theta = rng.uniform(-np.pi, np.pi, size=sample_count)
q = rng.normal(size=sample_count)
radius = 1.0 + 0.18 * rng.normal(size=sample_count)
cos_theta = np.cos(theta)
sin_theta = np.sin(theta)
tanh_q = np.tanh(q)
q_squared = q**2 - 1.0
basis = np.column_stack([
    radius * cos_theta, radius * sin_theta,
    np.cos(2 * theta), np.sin(2 * theta), np.cos(3 * theta), np.sin(3 * theta),
    tanh_q, q_squared, np.sin(1.5 * q), np.cos(1.5 * q),
    cos_theta + 0.55 * tanh_q, sin_theta - 0.55 * tanh_q,
    np.cos(2 * theta) + 0.35 * q_squared, np.sin(2 * theta) - 0.35 * q_squared,
    tanh_q * cos_theta, tanh_q * sin_theta, q_squared * cos_theta, q_squared * sin_theta,
    np.sin(theta + q), np.cos(theta - q),
])
confounders = basis + rng.normal(0.0, 0.12, size=basis.shape)
noise = rng.normal(size=(sample_count, 20))
synthetic_features = [f"feature_{index:02d}" for index in range(50)]
synthetic = pd.DataFrame(np.column_stack([spatial, confounders, noise]), columns=synthetic_features)
synthetic["x_m"] = coordinates[:, 0]
synthetic["y_m"] = coordinates[:, 1]

In [ ]:
synthetic_times = check_runtime("Synthetic", synthetic, synthetic_features, ["x_m", "y_m"])
display(synthetic_times.round({"seconds": 2}))

## 6. All runtimes
One row per dataset and method. Empty errors mean successful fits; failed fits have no runtime. Rerunning a dataset cell makes a new measurement and replaces that dataset's table.

In [ ]:
runtime_results = pd.concat([cmio_times, north_times, synthetic_times], ignore_index=True)
display(runtime_results.round({"seconds": 2}))